# 6 · Bronze, from files. And the first thing that is not a fact.

Two pipelines in one notebook, because the second one is short and the contrast
is the lesson.

| | |
|---|---|
| **p5** reads | gzipped CSV files in object storage (MinIO, which speaks S3) |
| **p5** writes | `teach.bronze_regulator` |
| **p6** reads | `kerb.zones`, a small reference table |
| **p6** writes | `teach.bronze_zones` |

The regulator does not have an API. Once a night they drop a file in a bucket
and that is the whole integration. This is far more common than anybody admits.

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import show, sql, fetch, run, counts

import csv, gzip, io, os, re
import psycopg
from minio import Minio
from pipelines.lib.config import dsn, SCHEMA

BUCKET, PREFIX = 'kerb-landing', 'regulator/'

def open_bucket():
    # MinIO speaks the S3 API, so this is the same code you would write against
    # real S3 with a different endpoint. Nothing here is toy.
    return Minio(
        os.environ['MINIO_ENDPOINT'].replace('http://', '').replace('https://', ''),
        access_key=os.environ['MINIO_ACCESS_KEY'],
        secret_key=os.environ['MINIO_SECRET_KEY'],
        secure=False)          # http, because it is running on this laptop

client = open_bucket()
print('buckets:', [b.name for b in client.list_buckets()])

---

## Step 0 · What is actually in the bucket?

In [ ]:
objects = list(client.list_objects(BUCKET, prefix=PREFIX, recursive=True))

# sorting by name works because the date is in the path in ISO order. That is
# not luck: whoever chose dt=YYYY-MM-DD made this sort correct.
objects.sort(key=lambda o: o.object_name, reverse=True)

print(f'{len(objects)} objects under {PREFIX!r}\n')
for o in objects[:7]:
    print(f'  {o.object_name:<58} {o.size / 1024:>8,.1f} KB')

### Two things to notice in those paths

**The date is in the path**, not in the file. `dt=2026-08-19/` is a convention
called partitioning, and it is why sorting by name sorts by date.

**Never list a bucket without a prefix.** A production landing bucket has
millions of objects, and listing all of them to find yesterday's is how you get
a very slow pipeline and a very large bill. The prefix filters on the server.

---

## Step 1 · Open one file and look at it

It is gzipped, so there are two steps before you have text.

In [ ]:
newest = objects[0]

raw  = client.get_object(BUCKET, newest.object_name).read()
text = gzip.decompress(raw).decode('utf-8')

print(f'{newest.object_name}')
print(f'{len(raw):,} bytes compressed  ->  {len(text):,} bytes of text\n')
print('\n'.join(text.splitlines()[:4]))

## Step 2 · Get the date out of the path

In [ ]:
PARTITION = re.compile(r'dt=(\d{4}-\d{2}-\d{2})/')

match = PARTITION.search(newest.object_name)
partition_date = match.group(1)
print('partition date:', partition_date)

# and a file in an unexpected place, which we must not guess a date for
print('a stray file  :', PARTITION.search('regulator/oops/data.csv.gz'))

**We parse the date from the path, never from `now()`.** A file that arrives
three days late is still Tuesday's file, and everything downstream depends on
that being true.

---

## Step 3 · Everything in a CSV is a string

This is where file pipelines actually break.

![](img/files-1-types.png)

In [ ]:
print('int("9")    ->', int('9'))

try:
    int('9.0')
except ValueError as e:
    print('int("9.0")  -> ValueError:', e)

The regulator writes zone ids as `9.0`, because whatever produced the file held
them as floats. So we go through `float` first.

In [ ]:
def as_int(value):
    """'9.0' -> 9, '9' -> 9, '' -> None. Never raises."""
    if value is None or value == '':
        return None
    return int(float(value))       # float() first: int('9.0') raises

def as_num(value):
    if value is None or value == '':
        return None
    return float(value)

for v in ['9.0', '9', '', None, '12.75']:
    print(f'  as_int({v!r:8}) = {as_int(v)!r:8}   as_num({v!r:8}) = {as_num(v)!r}')

---

## Step 4 · Read the file into rows, holding the ones that will not convert

One malformed row in a file of a thousand must not throw away the other nine
hundred and ninety nine.

In [ ]:
rows, held = [], []

for line_no, rec in enumerate(csv.DictReader(io.StringIO(text)), start=2):
    try:
        rows.append((
            partition_date,
            rec['trip_id'],
            as_int(rec['pu_zone_id']),
            as_int(rec['do_zone_id']),
            as_num(rec['distance_km']),
            as_int(rec['duration_s']),
            rec['status'],
            newest.object_name))          # keep the file this row came from
    except Exception as e:
        held.append((line_no, f'{type(e).__name__}: {e}'))

print(f'read   {len(rows) + len(held):,} lines')
print(f'landed {len(rows):,}')
print(f'held   {len(held):,}\n')
for r in rows[:3]:
    print(' ', r)

### That last column is the one people forget

`source_file` is the column that saves you. When a number looks wrong in six
weeks, *"which file did this row come from"* is the first question anybody asks.

Without it the answer is a shrug. With it you open that exact object and look.

---

## Step 5 · The table, and a key that admits the same ride twice

In [ ]:
DDL = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.bronze_regulator (
    partition_date DATE NOT NULL,    -- parsed from the path, not from the file
    trip_id        TEXT NOT NULL,
    pu_zone_id     INT,
    do_zone_id     INT,
    distance_km    NUMERIC(8,3),
    duration_s     INT,
    status         TEXT,
    source_file    TEXT NOT NULL,    -- exactly which object this row came from
    PRIMARY KEY (partition_date, trip_id)
);
"""

with psycopg.connect(dsn(), autocommit=True) as c:
    c.execute(DDL)

INSERT = f"""
    INSERT INTO {SCHEMA}.bronze_regulator
        (partition_date, trip_id, pu_zone_id, do_zone_id, distance_km,
         duration_s, status, source_file)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (partition_date, trip_id) DO NOTHING
"""

with psycopg.connect(dsn(), autocommit=False) as c, c.cursor() as cur:
    cur.executemany(INSERT, rows)
    c.commit()

print(f'wrote {len(rows):,} rows from one file')

The primary key is **`(partition_date, trip_id)`**, not `trip_id` alone. The
regulator can send the same ride on two different days, and **both are real**.
Keying on `trip_id` alone would silently throw one away.

---

## Step 6 · No file is not the same as an empty file

![](img/files-2-missing.png)

So count **files** as well as rows.

In [ ]:
from collections import Counter

per_day = Counter()
for o in objects:
    m = PARTITION.search(o.object_name)
    if m:
        per_day[m.group(1)] += 1

for day in sorted(per_day, reverse=True)[:7]:
    print(f'  {day}   {per_day[day]} file(s)')

missing = [d for d in sorted(per_day) if per_day[d] == 0]
print(f'\ndays with no file at all: {len(missing)}')

---

## Now the packaged file pipeline, over the last seven days

In [ ]:
run('-m', 'pipelines.p5_bronze_regulator', '--days', '7')

In [ ]:
sql(f"""
    SELECT partition_date, count(*) AS rides, count(DISTINCT source_file) AS files
    FROM {SCHEMA}.bronze_regulator
    GROUP BY 1 ORDER BY 1 DESC
""", 'what landed, by day and by file')

---

# The second pipeline, and the first thing that is not a fact

Everything so far has been **a fact**: something that happened, with a date.

`kerb.zones` is not that. It is a list of the sixty one zones the city is
divided into. It has no date because it is not about a moment, it is about **what
is true right now**.

![](img/files-3-dimension.png)

In [ ]:
sql("""
    SELECT zone_id, zone_name, borough, zone_type
    FROM kerb.zones ORDER BY zone_id LIMIT 6
""", 'kerb.zones, the dimension')

print()
print('rows in the dimension:', fetch('SELECT count(*) AS n FROM kerb.zones').n[0])

## Replace the whole thing, inside one transaction

No window. No partition. **Delete everything, insert everything, commit once.**

In [ ]:
DDL = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.bronze_zones (
    zone_id     INT PRIMARY KEY,
    zone_name   TEXT,
    borough     TEXT,
    zone_type   TEXT
);
"""                      # note: no date column anywhere in that table

with psycopg.connect(dsn(), autocommit=True) as c:
    c.execute(DDL)

with psycopg.connect(dsn()) as c:
    zones = c.execute("""SELECT zone_id, zone_name, borough, zone_type
                        FROM kerb.zones ORDER BY zone_id""").fetchall()

with psycopg.connect(dsn(), autocommit=False) as c, c.cursor() as cur:
    cur.execute(f'DELETE FROM {SCHEMA}.bronze_zones')
    cur.executemany(f'INSERT INTO {SCHEMA}.bronze_zones '
                    f'(zone_id, zone_name, borough, zone_type) VALUES (%s,%s,%s,%s)', zones)
    c.commit()

print(f'replaced the whole dimension: {len(zones)} rows')

### Why deleting everything is fine here, and would be madness in bronze_trips

Sixty one rows. Replacing all of them costs nothing.

`bronze_trips` has eighty four thousand rows for one week and millions in total,
which is exactly why it is rebuilt **one window at a time**.

**The transaction still matters**, for the same reason it always does: a reader
who queries halfway through must never see an empty table.

### And what a full snapshot loses

**History.** If a zone is renamed, the old name is gone and nothing records that
it ever existed.

That is usually fine for reference data. When it is not, the technique you reach
for is called a **slowly changing dimension**. Worth having heard the phrase.
Not worth building today.

In [ ]:
run('-m', 'pipelines.p6_bronze_zones')

In [ ]:
counts()

---

## What you learned

- Files are a real integration. Plenty of partners have nothing else
- **The date lives in the path.** Parse it from there, never from `now()`
- Always list a bucket **with a prefix**
- **A CSV has no types.** `float()` before `int()`, and wrap every conversion
- Keep **`source_file`**. It is the column that answers the six week old question
- **No file and empty file are different events.** Count files as well as rows
- A **fact** has a date and is rebuilt by window.
  A **dimension** has no date and is replaced whole
- Replacing a dimension loses history. The fix has a name: slowly changing dimension